
# Automatic BB

### Image comparing

In [5]:
import cv2

img_path1 = "editing/samples/sk11.png"
img_path2 = "editing/samples/sk22.png"
gen_img1 = "editing/samples/gen_img11.png"
gen_img2 = "editing/samples/gen_img22.png"

%load_ext autoreload
%autoreload 2

In [11]:
img1 = cv2.imread(img_path1, cv2.IMREAD_COLOR)
img2 = cv2.imread(img_path2, cv2.IMREAD_COLOR)

diff = cv2.absdiff(img1, img2)  # per-pixel absolute difference
cv2.imwrite("editing/samples/diff1.png", diff)

True

In [14]:
import cv2

# img_path1 = "editing/samples/image_1.png"
# img_path2 = "editing/samples/image_2.png"

img1 = cv2.imread(img_path1, cv2.IMREAD_COLOR)
img2 = cv2.imread(img_path2, cv2.IMREAD_COLOR)

# 1. Get the absolute per-pixel difference
diff = cv2.absdiff(img1, img2)

# 2. Convert the difference to grayscale
# This merges the BGR channels into a single intensity map so we can threshold it cleanly.
gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

# 3. Define your threshold value (0 to 255)
# A value of 30 is a great starting point to ignore tiny sensor noise or jpeg compression.
# Increase this number if it's picking up too much background noise.
threshold_value = 200 

# 4. Apply the Binary Threshold
# If the difference is > 30, force it to 255 (Pure White). 
# If it's <= 30, force it to 0 (Pure Black).
_, thresh_mask = cv2.threshold(gray_diff, threshold_value, 255, cv2.THRESH_BINARY)

# Save both so you can compare the raw difference vs the clean threshold mask
cv2.imwrite("editing/samples/img_diff1_raw.png", diff)
cv2.imwrite("editing/samples/diff1.png", thresh_mask)

True

In [14]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from depth_anything_3.api import DepthAnything3
import cv2
import torch
import time
import numpy as np

class DA3:
    def __init__(self):
        # model = DepthAnything3.from_pretrained("depth-anything/DA3MONO-LARGE")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        model = DepthAnything3.from_pretrained("depth-anything/DA3NESTED-GIANT-LARGE-1.1")
        # model = DepthAnything3.from_pretrained("depth-anything/DA3METRIC-LARGE")
        model = model.to(device)
        model.eval()
        print(f"Model loaded on {device}")
        
        self.run_times = []
        self.model = model
        self.device = device

    def forward(self, img_path):
        start = time.perf_counter()

        original_image = cv2.imread(img_path)
        H_orig, W_orig = original_image.shape[:2]
        longest_edge = max(H_orig, W_orig)
        optimal_res = int(round(longest_edge / (14.0)) * 14)
        process_res = min(optimal_res, 1330)
        print(f"Running native inference at process_res: {process_res}")

        prediction = self.model.inference(image=[img_path], process_res=process_res)

        depth = prediction.depth[0] # Depth in [m].
        print("depth:", depth.shape)

        depth_resized = cv2.resize(
            depth, 
            (W_orig, H_orig), 
            interpolation=cv2.INTER_LINEAR  # Change to cv2.INTER_NEAREST if edges stretch in 3D
        )
        depth_resized = depth_resized.astype(np.float32)
        print("depth_resized:", depth_resized.shape)

        if prediction.intrinsics is None:
            return depth_resized, None, None, None, None, None, 

        H_pred, W_pred = depth.shape

        scale_x = W_orig / W_pred
        scale_y = H_orig / H_pred
        fx = prediction.intrinsics[0, 0, 0] * scale_x
        fy = prediction.intrinsics[0, 1, 1] * scale_y
        cx = prediction.intrinsics[0, 0, 2] * scale_x
        cy = prediction.intrinsics[0, 1, 2] * scale_y

        h, w = depth_resized.shape
        SENSOR_HEIGHT_MM = 24.0  # Standard full-frame sensor height
        focal_length = (fy / h) * SENSOR_HEIGHT_MM
        print("focal_length:", focal_length)

        end = time.perf_counter()
        runtime = round(end - start, 3)
        self.run_times.append(runtime)

        return depth_resized, focal_length, fx, fy, cx, cy

    def mean_runtime (self):
        arr = np.array(self.run_times)
        return arr.mean()

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


In [3]:
da3 = DA3()

[INFO ] using SwiGLU layer as FFN
[INFO ] using MLP layer as FFN
Model loaded on cuda


In [6]:
depth, focal_length, fx, fy, cx, cy = da3.forward(gen_img1)

Running native inference at process_res: 518
[INFO ] Processed Images Done taking 0.07244443893432617 seconds. Shape:  torch.Size([1, 3, 518, 518])
[INFO ] Model Forward Pass Done. Time: 5.334021329879761 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0020067691802978516 seconds
depth: (518, 518)
depth_resized: (512, 512)
focal_length: 58.219536151665054


In [7]:
from editing.reconstruction import extract_changes

BB_Corners, changed_pcd = extract_changes(img_path1, img_path2, gen_img1, depth, fx, fy, cx, cy)
BB_Corners

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Pipeline complete. Files saved to 'editing/output'.


{'corners': [[-0.06470021495165207, 1.3575087807138635, -9.361572932087446],
  [1.3745610301402336, 2.0876934920674644, -12.845541535487127],
  [0.6140322983572069, -0.3048308672175669, -9.42958203916608],
  [-1.0186014725652437, 0.9873273316785971, -9.833222976418059],
  [1.0993922858355007, 0.05517239510076746, -13.385200686896374],
  [-0.3398689592563848, -0.6750123162528333, -9.901232083496692],
  [0.42065977252664194, 1.717512043032198, -13.31719157981774],
  [2.0532935434490924, 0.42535384413603383, -12.913550642565761]]}

In [8]:
from editing.test import run_alignment

M = run_alignment("kv_cache/mesh_pass1.glb", "editing/output/bg_free_full_pc.ply")
M

array([[ 0.1479807 ,  0.05088981, -0.16852144, -1.82062011],
       [-0.04109947,  0.2240573 ,  0.03157049,  0.28019424],
       [ 0.17117266,  0.00980254,  0.15326893,  1.55726494],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [10]:
from editing.reconstruction import transform_bounding_box, convert_bb_to_trellis

transformed_bb = transform_bounding_box(changed_pcd, M, padding=0.01)
# trellis_bb = convert_bb_to_trellis(transformed_bb)
transformed_bb

{'min': [-0.2531758031348228, -0.15229131178925426, -0.31077848019577714],
 'max': [0.6578392300897038, 0.24498879266196927, 0.14900512913521946]}

In [11]:
from editing.reconstruction import get_trellis_latent_mask

get_trellis_latent_mask(transformed_bb)

{'min': [0.34770868821074574, 0.35099487086478054, 0.2468241968651772],
 'max': [0.7449887926619693, 0.8107784801957771, 1.0]}

In [12]:
from editing.reconstruction import visualize_bounding_box

visualize_bounding_box("editing/output/bg_free_full_pc.ply", BB_Corners)

Visualizing: Oriented Bounding Box (Green)
Opening interactive viewer. Close the window to continue script execution.


In [13]:
visualize_bounding_box("editing/output/generated.ply", transformed_bb)

Visualizing: Axis-Aligned Bounding Box (Red)
Opening interactive viewer. Close the window to continue script execution.
